# Module 11: User Preference & Personalization Engine
## Style Profiles, Persona Presets & Adaptive Outfit Re-Ranking

This notebook demonstrates:
1. Defining granular fashion `UserProfile` objects (color affinities, categories, occasions, interaction history).
2. Standard persona presets (`Minimalist`, `Smart Casual`, `Vibrant Eclectic`, `Athletic Streetwear`).
3. Calculating individual item affinity and personalized outfit cohesion scores.
4. Comparing personalized recommendations across contrasting user personas.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns

from src.embeddings import EmbeddingManager
from src.style_matcher import StyleMatcher
from src.recommender import OutfitRecommender
from src.personalization import (
    UserProfile,
    PersonalizedRecommender,
    create_preset_profile,
    STYLE_ARCHETYPES,
)

sns.set_theme(style="white", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)

### 1. Initialize Personalization System

In [ ]:
mgr = EmbeddingManager()
matcher = StyleMatcher(mgr)
outfit_engine = OutfitRecommender(style_matcher=matcher)
pers_engine = PersonalizedRecommender(outfit_recommender=outfit_engine, personalization_weight=0.40)

print(f"Available Style Archetypes: {list(STYLE_ARCHETYPES.keys())}")
print(f"Personalization Strength (alpha): {pers_engine.alpha}")

### 2. Configure Distinct User Style Personas

In [ ]:
# Persona A: Minimalist Marcus (Monochrome, Clean, Neutrals)
user_a = create_preset_profile("Minimalist", user_id="user_marcus", gender="Men")

# Persona B: Smart Casual Sophia (Elevated, Navy/Beige/Pastels)
user_b = create_preset_profile("Smart Casual", user_id="user_sophia", gender="Women")

print("Persona A (Minimalist Marcus):")
print(f"  Favorite Colors: {user_a.favorite_colors}")
print(f"  Disliked Colors: {user_a.disliked_colors}")
print("\nPersona B (Smart Casual Sophia):")
print(f"  Favorite Colors: {user_b.favorite_colors}")
print(f"  Preferred Categories: {user_b.preferred_categories}")

### 3. Compare Personalized Outfit Recommendations

In [ ]:
outfits_a = pers_engine.recommend_personalized_outfits(user_a, occasion="Casual", top_k=2)

print(f"Top Recommended Outfit for Minimalist Marcus:")
top_a = outfits_a[0]
print(f"Personalized Score: {top_a['personalized_score']:.3f} | User Affinity: {top_a['user_affinity']:.3f}")
for it in top_a["items"]:
    print(f"  [{it['outfit_part'].upper():9s}] {it['canonical_category']} | {it['baseColour']} | {it['productDisplayName']}")

### 4. Personalized Clothing Feed

Generates a bespoke shopping feed ranked by user visual affinity.

In [ ]:
feed_items = pers_engine.recommend_feed(user_a, target_part="top", top_k=5)

print(f"Personalized Feed for Marcus (Top Garments):")
for rank, item in enumerate(feed_items, 1):
    print(f"#{rank} [Affinity: {item['user_affinity']:.3f}] - {item['canonical_category']} ({item['baseColour']}) | {item['productDisplayName']}")